In [34]:
#import libraries
import pandas as pd
import numpy as np
import warnings
import joblib
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,roc_auc_score

In [35]:
#load dataset
df=pd.read_csv(r"C:\Users\USER\Downloads\Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [36]:
#EDA
df.columns
df.info
df.describe()
df.isna().sum() #no null values found
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

In [37]:
#clean the data(convert any string to numeric)
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"],errors="coerce")
df["TotalCharges"].dtype                               

dtype('float64')

In [38]:
#check class balance
df["Churn"].value_counts()
#moderate imbalance

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [39]:
#target column
df["Churn"]=df["Churn"].map({"Yes":1,"No":0})
y=df["Churn"]
y

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7043, dtype: int64

In [40]:
#features
x=df.drop(["Churn","customerID"],axis=1)
numeric_col=x.select_dtypes(include=["int64","float64"]).columns
caregorical_com=[col for col in x.columns if col not in numeric_col]
print(numeric_col)
print(caregorical_com)

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [41]:
#apply preprocessing seperately on numeric column and categorical column
numeric_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="mean")),
    ("scaler",StandardScaler())
])

categorical_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])

#use columnTransfer to apply preprocessing steps
preprocessor=ColumnTransformer([
    ("num", numeric_pipeline, numeric_col),
    ("cat", categorical_pipeline, caregorical_com)
])

In [42]:
#train model
model_pipeline=Pipeline([
    ("preprocessing",preprocessor),
    ("model",LogisticRegression(class_weight="balanced"))
])
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
model_pipeline.fit(x_train,y_train)

#save model
joblib.dump(model_pipeline,"model.pkl")

['model.pkl']

In [43]:
#accuracy
#train
y_pred_train=model_pipeline.predict(x_train)
train_acc=accuracy_score(y_train,y_pred_train)
print("Train accuracy",train_acc)
#test
y_pred_test=model_pipeline.predict(x_test)
test_acc=accuracy_score(y_test,y_pred_test)
print("Test accuracy",test_acc)

#confusion_matrix
matrix=confusion_matrix(y_test,y_pred_test)
print("confusion_matrix\n",matrix)

#classification_report
report=classification_report(y_test,y_pred_test)
print("classification_report\n",report)
    

Train accuracy 0.7442314518991835
Test accuracy 0.7487579843860894
confusion_matrix
 [[748 288]
 [ 66 307]]
classification_report
               precision    recall  f1-score   support

           0       0.92      0.72      0.81      1036
           1       0.52      0.82      0.63       373

    accuracy                           0.75      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409



In [44]:
#roc_auc_score
y_prob=model_pipeline.predict_proba(x_test)[:,1]
y_prob=(y_prob>0.4).astype(int)
roc_auc_score=roc_auc_score(y_test,y_prob)
print("roc_auc_score",roc_auc_score)

roc_auc_score 0.7571384578757232
